# Zerobus Ingest Producer for Healthcare Events

This notebook demonstrates near real-time data ingestion into Databricks Delta tables using the Zerobus Ingest SDK.

**What is Zerobus?**  
Zerobus allows you to stream data directly into Delta tables via gRPC - no Kafka or message bus needed!

**GitHub**: [bigdatavik/zerobus-healthcare-demo](https://github.com/bigdatavik/zerobus-healthcare-demo)

---
# DEMO RESET: Run This Before a Demo

Zerobus **APPENDS** data (it never truncates). Run this cell to reset the table before a demo.

**Note:** This deletes ALL data in the table!

In [ ]:
# Update TABLE_NAME below to match your table
DEMO_TABLE = "<catalog>.<schema>.healthcare_events"

# Check current record count
print("Current record count:")
spark.sql(f"SELECT COUNT(*) as count FROM {DEMO_TABLE}").show()

# Uncomment the line below to TRUNCATE (delete all data)
# spark.sql(f"TRUNCATE TABLE {DEMO_TABLE}")
# print("Table truncated! Record count is now 0.")

---
## Architecture Comparison

### Traditional Way (Complex)
```
┌──────────┐    ┌─────────┐    ┌─────────────┐    ┌────────────┐
│  Source  │───▶│  Kafka  │───▶│ Spark Job   │───▶│ Delta Lake │
└──────────┘    └─────────┘    └─────────────┘    └────────────┘
                     ↑               ↑
              Need to manage   Need to manage
```

### Zerobus Way (Simple)
```
┌──────────┐    ┌──────────────────┐    ┌────────────┐
│  Source  │───▶│  Zerobus (gRPC)  │───▶│ Delta Lake │
└──────────┘    └──────────────────┘    └────────────┘
                        ↑
                No infrastructure!
                Serverless & managed
```

## The 5 Key Lines That Do All the Network Work

| Line | Code | What Happens |
|------|------|-------------|
| 1 | `sdk = ZerobusSdk(SERVER_ENDPOINT, WORKSPACE_URL)` | Creates client object (no network yet) |
| 2 | `stream = sdk.create_stream(CLIENT_ID, CLIENT_SECRET, ...)` | Opens gRPC connection + authenticates |
| 3 | `ack = stream.ingest_record(record)` | Sends record over gRPC |
| 4 | `ack.wait_for_ack()` | Waits for confirmation (durability guarantee!) |
| 5 | `stream.close()` | Closes the gRPC connection |

## How to Create Credentials

### 1. Create a Service Principal
```bash
databricks service-principals create --display-name "zerobus-producer"
```
Returns: `applicationId` → This is your **CLIENT_ID**

### 2. Generate OAuth Secret
```bash
databricks service-principals secrets create <service-principal-id>
```
Returns: `secret` → This is your **CLIENT_SECRET**

### 3. Grant Permissions
```sql
GRANT USE CATALOG ON CATALOG <catalog> TO `<service-principal-id>`;
GRANT USE SCHEMA ON SCHEMA <catalog>.<schema> TO `<service-principal-id>`;
GRANT MODIFY, SELECT ON TABLE <catalog>.<schema>.<table> TO `<service-principal-id>`;
```

---
# Step 1: Install Dependencies

The Zerobus SDK needs to be installed. Run this cell or configure in your cluster/job environment.

In [ ]:
# Install Zerobus SDK
%pip install databricks-zerobus-ingest-sdk>=0.2.0

---
# Step 2: Imports

These are the only imports needed for Zerobus:
- **ZerobusSdk**: The main client class
- **RecordType**: Enum for serialization format (JSON or PROTOBUF)
- **StreamConfigurationOptions**: Settings for the stream
- **TableProperties**: Identifies the target Delta table

In [ ]:
import json
import time
import uuid
import random
from datetime import datetime

# Zerobus SDK imports
from zerobus.sdk.sync import ZerobusSdk
from zerobus.sdk.shared import RecordType, StreamConfigurationOptions, TableProperties

print("Imports successful!")

---
# Step 3: Configuration

### Zerobus Server Endpoint
Format: `<workspace_id>.zerobus.<region>.cloud.databricks.com`

**How to find your workspace_id:**
- Look at your workspace URL: `https://<name>.cloud.databricks.com/?o=<workspace_id>`
- Or run: `databricks workspace get-status /`

### Service Principal Credentials
- **CLIENT_ID**: The `applicationId` from the service principal
- **CLIENT_SECRET**: The OAuth secret generated for the service principal

In [ ]:
# =============================================================================
# CONFIGURE THESE VALUES FOR YOUR ENVIRONMENT
# =============================================================================

# Zerobus server endpoint: <workspace_id>.zerobus.<region>.cloud.databricks.com
SERVER_ENDPOINT = "<workspace_id>.zerobus.<region>.cloud.databricks.com"

# Databricks workspace URL
WORKSPACE_URL = "https://<your-workspace>.cloud.databricks.com"

# Target table (must exist in Unity Catalog)
TABLE_NAME = "<catalog>.<schema>.healthcare_events"

# Service principal credentials
CLIENT_ID = "<your-service-principal-client-id>"
CLIENT_SECRET = "<your-service-principal-secret>"

print(f"Server Endpoint: {SERVER_ENDPOINT}")
print(f"Target Table: {TABLE_NAME}")

---
# Step 4: Sample Data Generation

This section generates fake healthcare events for the demo.

## Real-World Architecture

In production, the producer runs **outside Databricks** (on-prem server, cloud VM, or container) and connects via the internet:

```
┌─────────────────────────────────────────────────────────────────────────────────┐
│                        HEALTHCARE DATA CENTER (On-Prem or Cloud)                │
│                                                                                 │
│  ┌─────────────┐   ┌─────────────┐   ┌─────────────┐   ┌─────────────┐         │
│  │    Epic     │   │   Cerner    │   │  Lab Info   │   │  Pharmacy   │         │
│  │    EHR      │   │    EHR      │   │   System    │   │   System    │         │
│  └──────┬──────┘   └──────┬──────┘   └──────┬──────┘   └──────┬──────┘         │
│         │                 │                 │                 │                │
│         └────────────┬────┴────────┬────────┴────────┬────────┘                │
│                      │             │                 │                         │
│                      ▼             ▼                 ▼                         │
│         ┌──────────────────────────────────────────────────────┐               │
│         │              INTEGRATION SERVER                       │               │
│         │    (Python app with Zerobus SDK installed)           │               │
│         │                                                       │               │
│         │    from zerobus.sdk.sync import ZerobusSdk           │               │
│         │    sdk = ZerobusSdk(SERVER_ENDPOINT, WORKSPACE_URL)  │               │
│         │    stream = sdk.create_stream(...)                    │               │
│         │    stream.ingest_record(event)  ◄─── YOUR CODE       │               │
│         └───────────────────────┬───────────────────────────────┘               │
│                                 │                                               │
└─────────────────────────────────┼───────────────────────────────────────────────┘
                                  │
                                  │ gRPC over HTTPS (port 443)
                                  │ (Internet / VPN / Direct Connect)
                                  │
                                  ▼
┌─────────────────────────────────────────────────────────────────────────────────┐
│                              DATABRICKS CLOUD                                    │
│                                                                                 │
│  ┌───────────────────────────────────────────────────────────────────────────┐ │
│  │                    ZEROBUS INGEST SERVICE                                  │ │
│  │                                                                            │ │
│  │    • Receives gRPC streams from anywhere                                   │ │
│  │    • Validates credentials (OAuth)                                         │ │
│  │    • Validates schema against target table                                 │ │
│  │    • Batches records for efficiency                                        │ │
│  │    • Writes to Delta Lake                                                  │ │
│  │    • Sends durability ACKs back to producer                               │ │
│  └───────────────────────────────────┬───────────────────────────────────────┘ │
│                                      │                                          │
│                                      ▼                                          │
│  ┌───────────────────────────────────────────────────────────────────────────┐ │
│  │                         UNITY CATALOG                                      │ │
│  │                                                                            │ │
│  │   <catalog>.<schema>.healthcare_events                                    │ │
│  │   ┌────────────────────────────────────────────────────────────────────┐  │ │
│  │   │  event_id │ member_id │ event_type │ event_timestamp │ amount │ ... │  │ │
│  │   ├───────────┼───────────┼────────────┼─────────────────┼────────┼─────┤  │ │
│  │   │  uuid-1   │ MBR123456 │ admission  │ 1705312800...   │ 12500  │     │  │ │
│  │   │  uuid-2   │ MBR789012 │ claim      │ 1705312801...   │ 350    │     │  │ │
│  │   └────────────────────────────────────────────────────────────────────┘  │ │
│  └───────────────────────────────────────────────────────────────────────────┘ │
│                                                                                 │
└─────────────────────────────────────────────────────────────────────────────────┘
```

## This Demo vs Production

| Aspect | This Demo | Production |
|--------|-----------|------------|
| **Where producer runs** | Inside Databricks (notebook/job) | External server (on-prem, EC2, K8s) |
| **Data source** | Random generated | Real EHR/Claims systems |
| **Connection** | Internal | Internet/VPN/DirectConnect |
| **Why?** | Simpler to demo | Real-world architecture |

## Important Notes:
- **event_timestamp**: Must be an integer (microseconds), NOT a string!
- **metadata**: Must be a JSON string, not a Python dict

In [ ]:
# Sample data options
EVENT_TYPES = ["admission", "discharge", "claim", "rx_fill", "lab_result"]
FACILITY_CODES = ["FAC001", "FAC002", "FAC003", "FAC004", "FAC005"]
DIAGNOSIS_CODES = ["E11.9", "I10", "J06.9", "M54.5", "K21.0", "F32.9", "J45.909"]  # ICD-10
PROCEDURE_CODES = ["99213", "99214", "99215", "99203", "99204", "90834", "96372"]  # CPT
PROVIDER_NPIS = ["1234567890", "2345678901", "3456789012", "4567890123", "5678901234"]

def generate_healthcare_event():
    """Generate a single sample healthcare event record."""
    event_type = random.choice(EVENT_TYPES)
    
    # Realistic amounts based on event type
    amount_ranges = {
        "admission": (5000, 50000),
        "discharge": (0, 500),
        "claim": (100, 10000),
        "rx_fill": (10, 500),
        "lab_result": (50, 1000)
    }
    min_amt, max_amt = amount_ranges.get(event_type, (100, 1000))
    
    # Build metadata
    metadata = {
        "source_system": "demo_producer",
        "ingestion_batch": str(uuid.uuid4())[:8]
    }
    
    if event_type == "admission":
        metadata["admission_type"] = random.choice(["emergency", "elective", "urgent"])
    elif event_type == "rx_fill":
        metadata["ndc_code"] = f"{random.randint(10000, 99999)}-{random.randint(100, 999)}-{random.randint(10, 99)}"
    elif event_type == "lab_result":
        metadata["test_type"] = random.choice(["CBC", "BMP", "HbA1c", "Lipid Panel"])
    
    return {
        "event_id": str(uuid.uuid4()),
        "member_id": f"MBR{random.randint(100000, 999999)}",
        "event_type": event_type,
        "event_timestamp": int(time.time() * 1_000_000),  # IMPORTANT: Microseconds!
        "facility_code": random.choice(FACILITY_CODES),
        "diagnosis_code": random.choice(DIAGNOSIS_CODES),
        "procedure_code": random.choice(PROCEDURE_CODES),
        "provider_npi": random.choice(PROVIDER_NPIS),
        "amount": round(random.uniform(min_amt, max_amt), 2),
        "metadata": json.dumps(metadata)  # IMPORTANT: Must be JSON string!
    }

# Test: Generate one event
sample_event = generate_healthcare_event()
print("Sample event:")
for key, value in sample_event.items():
    print(f"  {key}: {value}")

---
# Step 5: Initialize the Zerobus SDK

This creates a client object but does **NOT** connect yet.  
No network call happens here - it's just storing the endpoints.

```
┌─────────────────────────────────────────────────────┐
│  sdk object created in memory                       │
│  server_endpoint: "<workspace_id>.zerobus..."       │
│  workspace_url: "https://<workspace>..."            │
│  (No connection yet)                                │
└─────────────────────────────────────────────────────┘
```

In [ ]:
# Initialize SDK (no network call yet)
sdk = ZerobusSdk(SERVER_ENDPOINT, WORKSPACE_URL)

print("SDK initialized (no connection yet)")

---
# Step 6: Configure Stream Options

**RecordType options:**
- `RecordType.JSON` = Send records as JSON (simpler, good for demos)
- `RecordType.PROTOBUF` = Send as Protocol Buffers (faster, type-safe, production)

In [ ]:
# Configure for JSON serialization
options = StreamConfigurationOptions(record_type=RecordType.JSON)

# Point to our target table
table_props = TableProperties(TABLE_NAME)

print(f"Stream configured for: {TABLE_NAME}")
print(f"Serialization: JSON")

---
# Step 7: Create the Stream (FIRST NETWORK CALL!)

This is where the magic happens:
1. Opens a gRPC connection to the Zerobus server
2. Authenticates using OAuth (CLIENT_ID + CLIENT_SECRET)
3. Tells Zerobus which table we want to write to
4. Returns a stream object for sending records

```
YOUR CODE                              ZEROBUS SERVER
    │                                        │
    │  "I want to write to table X"          │
    │  "Here's my credentials: ..."          │
    │───────────────────────────────────────▶│
    │                                        │
    │            "OK, authenticated.         │
    │             Stream ready."             │
    │◀───────────────────────────────────────│
```

In [ ]:
# Create stream (THIS CONNECTS TO ZEROBUS!)
print("Connecting to Zerobus...")
stream = sdk.create_stream(CLIENT_ID, CLIENT_SECRET, table_props, options)
print("Stream created! Connected and authenticated.")

---
# Step 8: Ingest Records (SENDS DATA OVER gRPC!)

For each record:
1. `stream.ingest_record(record)` - Sends the record (returns immediately)
2. `ack.wait_for_ack()` - Waits for confirmation that it's written to Delta

## Where Do These Methods Come From?

These are **NOT** Python built-ins. They come from the **Databricks Zerobus SDK**:

```
┌─────────────────────────────────────────────────────────────────┐
│  pip install databricks-zerobus-ingest-sdk                      │
│                                                                 │
│  This installs the zerobus package which contains:              │
│                                                                 │
│  ┌───────────────────────────────────────────────────────────┐  │
│  │  ZerobusSdk class                                         │  │
│  │    └── .create_stream() → returns a Stream object         │  │
│  │                                                           │  │
│  │  Stream class                                             │  │
│  │    ├── .ingest_record(dict) → returns Ack object    ◄────┼──┼── SENDS DATA
│  │    └── .close()                                           │  │
│  │                                                           │  │
│  │  Ack class                                                │  │
│  │    └── .wait_for_ack() → blocks until confirmed     ◄────┼──┼── WAITS FOR ACK
│  └───────────────────────────────────────────────────────────┘  │
└─────────────────────────────────────────────────────────────────┘
```

**What `ingest_record()` does internally:**
1. Serializes your Python dict to JSON (or Protobuf)
2. Sends it over the gRPC connection
3. Returns an `Ack` object you can wait on

## The Durability Guarantee

After `wait_for_ack()` returns, you **KNOW**:
- The record is written to Delta Lake
- It's immediately queryable
- It's safe even if Zerobus crashes

```
YOUR CODE                          ZEROBUS SERVER
    │                                    │
    │  {record data as JSON}             │
    │───────────────────────────────────▶│  ← ingest_record()
    │                                    │
    │      (Zerobus writes to Delta...)  │
    │      (batches, commits, flushes)   │
    │                                    │
    │        "ACK - Record written!"     │
    │◀───────────────────────────────────│  ← wait_for_ack()
```

In [ ]:
# Ingest 10 healthcare events
num_records = 10
successful = 0
failed = 0

print(f"Ingesting {num_records} healthcare events...")
print("-" * 60)

for i in range(num_records):
    # Generate a sample event
    record = generate_healthcare_event()
    
    try:
        # Send the record over gRPC
        ack = stream.ingest_record(record)
        
        # Wait for durability confirmation
        ack.wait_for_ack()
        
        successful += 1
        print(f"  [{i+1}/{num_records}] {record['event_type']:12} | {record['member_id']} | ${record['amount']:,.2f}")
        
    except Exception as e:
        failed += 1
        print(f"  [{i+1}/{num_records}] FAILED: {e}")

print("-" * 60)
print(f"Ingestion complete! Successful: {successful}, Failed: {failed}")

---
# Step 9: Close the Stream (CLEANUP)

Always close the stream when done:
1. Flushes any remaining buffered records
2. Closes the gRPC connection
3. Releases resources

In [ ]:
# Close the stream
stream.close()
print("Stream closed. Done!")

---
# Step 10: Validate Your Data

Run these SQL queries to verify the data was ingested.

**Note:** Update the table name to match your configuration.

In [ ]:
# Count total records
df = spark.sql(f"SELECT COUNT(*) as total_records FROM {TABLE_NAME}")
df.show()

In [ ]:
# View recent records
df = spark.sql(f"""
    SELECT 
        event_id,
        member_id,
        event_type,
        from_unixtime(event_timestamp / 1000000) as event_time,
        amount
    FROM {TABLE_NAME}
    ORDER BY event_timestamp DESC
    LIMIT 10
""")
df.show(truncate=False)

In [ ]:
# Analytics by event type
df = spark.sql(f"""
    SELECT 
        event_type,
        COUNT(*) as count,
        ROUND(AVG(amount), 2) as avg_amount,
        ROUND(SUM(amount), 2) as total_amount
    FROM {TABLE_NAME}
    GROUP BY event_type
    ORDER BY count DESC
""")
df.show()

---
# Summary

## What We Did
1. **Initialized SDK** - Created client object (no network)
2. **Created Stream** - Opened gRPC connection, authenticated
3. **Ingested Records** - Sent data, waited for ACKs
4. **Closed Stream** - Cleaned up resources
5. **Validated Data** - Queried the Delta table

## Key Takeaways
- **No Kafka** - Direct writes to Delta Lake
- **Durability Guarantee** - `wait_for_ack()` confirms data is written
- **Simple SDK** - Only 5 lines of network code
- **Immediate Queryability** - Data is available instantly after ACK
- **Append Only** - Zerobus always appends, use TRUNCATE to reset

## Learn More
- [Zerobus Documentation](https://docs.databricks.com/aws/en/ingestion/zerobus-overview)
- [GitHub Repository](https://github.com/bigdatavik/zerobus-healthcare-demo)